# A Reconstruction Pipeline using RGBD images and Alignement

## Step 0 - Define Commons

In [1]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from transformers import pipeline
import torch.nn.functional as F
import open3d as o3d
import os
import cv2
import sys
sys.path.append("../")
from utils.imageSelector import *

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# --- Configure paths ---
input_dir = Path("../data/images")
output_dir = Path("../results/rgbd_turntable/depth_images")
output_dir.mkdir(parents=True, exist_ok=True)

# --- Load Depth Anything v2 ---
device = "cuda" if torch.cuda.is_available() else "cpu"

model_candidates = [
    "depth-anything/Depth-Anything-V2-Small-hf",
    "depth-anything/Depth-Anything-V2-Base-hf",
    "depth-anything/Depth-Anything-V2-Large-hf",
]

""" pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf"  # or Large
) """

# --- Process all images in folder ---
valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
image_paths = [p for p in sorted(input_dir.iterdir()) if p.suffix.lower() in valid_ext]

## Step 1 - Generate Depth Images 

In [ ]:
def normalize_depth_by_mask(depth_relative, object_mask):
    """
    Normalize relative depth within the object region only.
    Produces consistent shape across frames even without metric scale.
    
    object_mask: boolean H x W, True where object pixels are
    """
    obj_depths = depth_relative[object_mask]
    
    d_min, d_max = obj_depths.min(), obj_depths.max()
    if d_max - d_min < 1e-6:
        return np.zeros_like(depth_relative)
    
    # Normalize to [0, 1] within object, zero outside
    normalized = np.where(
        object_mask,
        (depth_relative - d_min) / (d_max - d_min),
        0.0
    )
    return normalized

In [3]:
processor, model = None, None
for model_id in model_candidates:
    try:
        processor = AutoImageProcessor.from_pretrained(model_id)
        model = AutoModelForDepthEstimation.from_pretrained(model_id).to(device).eval()
        print(f"Loaded model: {model_id}")
        break
    except Exception:
        continue

if model is None:
    raise RuntimeError("Could not load a Depth Anything v2 model from the candidate list.")

if not image_paths:
    print(f"No images found in: {input_dir}")
else:
    for img_path in image_paths:
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            pred = outputs.predicted_depth  # [B, H', W']

        # Upscale depth to original image size
        pred_up = F.interpolate(
            pred.unsqueeze(1), size=(h, w), mode="bicubic", align_corners=False
        ).squeeze().cpu().numpy()

        # Normalize to 16-bit depth image
        d_min, d_max = pred_up.min(), pred_up.max()
        if d_max > d_min:
            depth_u16 = ((pred_up - d_min) / (d_max - d_min) * 65535.0).astype(np.uint16)
        else:
            depth_u16 = np.zeros((h, w), dtype=np.uint16)

        out_path = output_dir / f"{img_path.stem}_depth.png"
        Image.fromarray(depth_u16).save(out_path)

    print(f"Saved {len(image_paths)} depth images to: {output_dir}")

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loaded model: depth-anything/Depth-Anything-V2-Small-hf
Saved 112 depth images to: ..\results\rgbd_turntable\depth_images


## Step 2 - Define Camera Intrinsics

This can be done through calibration with a calibration pad

In [4]:
fx = 1.73385664e+03  # focal length x (pixels)
fy = 1.76271719e+03  # focal length y (pixels)
cx = 5.41424692e+02  # principal point x
cy = 9.15582474e+02  # principal point y
width, height = 1080, 1280

## Step 3 - Back-Project depth frames into 3D

In [32]:
def relative_to_metric(depth_rel, camera_radius, background_dist):
    """
    Convert pre-masked relative depth to metric depth.

    depth_rel       : H x W float, masked relative depth (invalid/background <= 0)
    camera_radius   : known distance to object center (meters)
    background_dist : far anchor distance (meters)
    """
    valid = np.isfinite(depth_rel) & (depth_rel > 0)
    if valid.sum() == 0:
        return np.zeros_like(depth_rel, dtype=np.float32)

    vals = depth_rel[valid]
    p_near = np.median(vals)   # object anchor
    p_far = np.min(vals)       # far anchor among valid pixels

    if abs(p_near - p_far) < 1e-6:
        return np.zeros_like(depth_rel, dtype=np.float32)

    A = np.array([[p_near, 1.0],
                  [p_far,  1.0]], dtype=np.float32)
    rhs = np.array([1.0 / camera_radius,
                    1.0 / background_dist], dtype=np.float32)

    a, b = np.linalg.solve(A, rhs)

    disparity = a * depth_rel + b
    metric = np.where(valid & (disparity > 0), 1.0 / disparity, 0.0)
    return metric.astype(np.float32)

In [33]:
def depth_to_pointcloud(color, depth_metric, fx, fy, cx, cy):
    h, w = depth_metric.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))

    Z = depth_metric
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    valid = Z > 0
    points = np.stack([X[valid], Y[valid], Z[valid]], axis=-1)
    colors = color[valid].astype(np.float32) / 255.0
    return points, colors

In [34]:
points = []
colors = []

for img_path in image_paths:
    color_img = np.array(Image.open(img_path).convert("RGB"))
    depth_img = np.array(Image.open(output_dir / f"{Path(img_path).stem}_depth.png"))
    point, color = depth_to_pointcloud(color_img, depth_img, fx, fy, cx, cy)
    points.append(point)
    colors.append(color)

## Step 4 - Compute Camera Poses

In [35]:
def get_camera_pose(frame_idx, num_frames, radius, elevation):
    angle = -2 * np.pi * frame_idx / num_frames   # negate: opposite to CW turntable

    cam_pos = np.array([
        radius * np.sin(angle),
        elevation,
        radius * np.cos(angle)
    ])

    forward = -cam_pos / np.linalg.norm(cam_pos)
    up      = np.array([0.0, 1.0, 0.0])
    right   = np.cross(forward, up);  right /= np.linalg.norm(right)
    up      = np.cross(right, forward)

    R = np.array([right, -up, forward])
    t = -R @ cam_pos

    T = np.eye(4)
    T[:3, :3] = R
    T[:3,  3] = t
    return T


def transform_to_world(pts_cam, T):
    R, t = T[:3, :3], T[:3, 3]
    cam_origin = -R.T @ t
    return pts_cam @ R + cam_origin

In [37]:
camera_poses = []
num_frames = len(image_paths)
for i in range(num_frames):
    pose = get_camera_pose(i, num_frames, 0.5, 0.3)
    camera_poses.append(pose)

## Step 5 - Transform points to World Space and Merge

In [42]:
def load_frame(color_path, depth_path):
    color = np.array(Image.open(color_path).convert("RGB"))
    depth = np.array(Image.open(depth_path))   # or cv2.imread(..., cv2.IMREAD_ANYDEPTH)
    return color, depth

def scale_intrinsics(fx, fy, cx, cy, scale: float):
    """
    Scale camera intrinsics to match a downsampled image.
    All four parameters scale linearly with resolution.
    """
    return fx * scale, fy * scale, cx * scale, cy * scale

def downsample_rgbd(color_img, depth_img, scale: float,
                    depth_interpolation=cv2.INTER_NEAREST):
    """
    Downsample a color + depth image pair and return scaled intrinsics.
    """
    h, w = depth_img.shape
    new_w = int(w * scale)
    new_h = int(h * scale)
    new_size = (new_w, new_h)   # cv2 wants (width, height)

    color_small = cv2.resize(color_img, new_size, interpolation=cv2.INTER_LINEAR)

    # INTER_NEAREST is critical for depth: avoids invented depth values
    # at edges where foreground meets background
    depth_small = cv2.resize(depth_img, new_size, interpolation=depth_interpolation)

    return color_small, depth_small, (new_w, new_h)

def build_point_cloud(frame_paths, num_frames, fx, fy, cx, cy,
                      depth_scale=1000.0, radius=0.5):
    all_points = []
    all_colors = []

    for i, (color_path, depth_path) in enumerate(frame_paths):
        
        color, depth = load_frame(color_path, depth_path)

        color, depth, (w, h) = downsample_rgbd(color, depth, scale=0.25)

        fx_s, fy_s, cx_s, cy_s = scale_intrinsics(fx, fy, cx, cy, 0.25)

        depth_metric = relative_to_metric(depth, 0.3, 0.5)

        # 1. Back-project to camera space
        pts_cam, cols = depth_to_pointcloud(color, depth, fx_s, fy_s, cx_s, cy_s)

        # 2. Get camera extrinsics
        T = get_camera_pose(i, num_frames, 0.3, 0)
        pts_world = transform_to_world(pts_cam, T)

        all_points.append(pts_world)
        all_colors.append(cols)

        print(f"Frame {i+1}/{num_frames} — {len(pts_world):,} points")

    points = np.vstack(all_points)
    colors = np.vstack(all_colors)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    return pcd

## Step 6 - Post-processing and Save

In [43]:
# Build it
image_paths = (select_equally_distributed_images(input_dir, n=36)) # e.g. 36 images → every 10°
frame_paths = [(Path(img_path), output_dir / f"{Path(img_path).stem}_depth.png") for img_path in image_paths]

pcd = build_point_cloud(frame_paths, num_frames=36,   # e.g. 10° steps
                        fx=fx, fy=fy, cx=cx, cy=cy,
                        depth_scale=0.1, radius=0.5)

print(f"Combined point cloud has {len(pcd.points):,} points before filtering")
print("Removing outliers and downsampling...")

# Remove statistical outliers (noise)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Optional: voxel downsample to reduce density uniformly
pcd = pcd.voxel_down_sample(voxel_size=0.002)   # 2mm voxels

print(f"Point cloud has {len(pcd.points):,} points after filtering and downsampling")
print("Estimating normals...")

# Estimate normals (needed for meshing later)
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.01, max_nn=30))
print("Normals estimated.")

# Save
o3d.io.write_point_cloud("../results/rgbd_turntable/Reconstruction.ply", pcd)
print(f"Saved {len(pcd.points):,} points → Reconstruction.ply")

# Visualize
o3d.visualization.draw_geometries([pcd])

Frame 1/36 — 129,600 points
Frame 2/36 — 129,600 points
Frame 3/36 — 129,600 points
Frame 4/36 — 129,600 points
Frame 5/36 — 129,600 points
Frame 6/36 — 129,600 points
Frame 7/36 — 129,600 points
Frame 8/36 — 129,600 points
Frame 9/36 — 129,600 points
Frame 10/36 — 129,600 points
Frame 11/36 — 129,599 points
Frame 12/36 — 129,600 points
Frame 13/36 — 129,600 points
Frame 14/36 — 129,600 points
Frame 15/36 — 129,600 points
Frame 16/36 — 129,600 points
Frame 17/36 — 129,600 points
Frame 18/36 — 129,600 points
Frame 19/36 — 129,600 points
Frame 20/36 — 129,600 points
Frame 21/36 — 129,600 points
Frame 22/36 — 129,599 points
Frame 23/36 — 129,600 points
Frame 24/36 — 129,600 points
Frame 25/36 — 129,600 points
Frame 26/36 — 129,600 points
Frame 27/36 — 129,600 points
Frame 28/36 — 129,600 points
Frame 29/36 — 129,600 points
Frame 30/36 — 129,600 points
Frame 31/36 — 129,600 points
Frame 32/36 — 129,600 points
Frame 33/36 — 129,600 points
Frame 34/36 — 129,600 points
Frame 35/36 — 129,600 p

## Step 7 - Making Mesh

In [ ]:
volume = o3d.pipelines.integration.ScalableTSDFVolume(
    voxel_length=0.004,
    sdf_trunc=0.02,
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8)

for i, (color_path, depth_path) in enumerate(frame_paths):
    color, depth = load_frame(color_path, depth_path)
    T = get_camera_pose(i, num_frames, 0.5)
    
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        o3d.geometry.Image(color),
        o3d.geometry.Image(depth.astype(np.uint16)),
        depth_scale=1000.0, depth_trunc=1.5, convert_rgb_to_intensity=False)
    
    intrinsic = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)
    volume.integrate(rgbd, intrinsic, T)

mesh = volume.extract_triangle_mesh()
mesh.compute_vertex_normals()
o3d.io.write_triangle_mesh(os.path.join(output_dir,"output_mesh.ply"), mesh)